# What this notebook is doing (Untitled2 / CONUS404 precip prep)

This notebook is basically a working copy of the CONUS404 precipitation extraction workflow for Skagit.
It pulls CONUS404 precipitation through the intake catalog, clips it to the Skagit boundary, converts the data into daily precipitation in mm, and saves the result as a local derived Zarr file.

The output includes both the basin-mean daily series and the native-grid daily precipitation with the basin mask.
So this is more of a preprocessing / data-prep notebook than a final analysis notebook.

It overlaps a lot with the CONUS downloader notebook, so I would treat this as another working version of that same idea rather than a separate finished workflow.


In [ ]:
# ========================= CONUS404 → Skagit daily precip (Zarr v2) =========================
# Defensive Intake walk (skip parquet/missing plugins), increments→mm, basin+grid,
# Zarr v2 with uniform chunking, two-phase write (series first, then grid+mask)
# ============================================================================================

import os, json, shutil
from pathlib import Path

import numpy as np
import xarray as xr
import intake

# ---------------- Config ----------------
CAT_URL      = "https://raw.githubusercontent.com/hytest-org/hytest/main/dataset_catalog/hytest_intake_catalog.yml"
BOUNDARY_GEO = Path("../data/GIS/SkagitBoundary.json")    # GeoJSON Feature / FeatureCollection / raw geometry
OUT_ZARR     = Path("/data0/balaji24/data/derived/conus404_skagit_precip_daily.zarr")

YEAR_MIN, YEAR_MAX = 2014, 2020

# ---------------- Helpers ----------------
def read_geom(path: Path):
    obj = json.load(open(path))
    if isinstance(obj, dict) and obj.get("type") == "FeatureCollection":
        return obj["features"][0]["geometry"]
    if isinstance(obj, dict) and obj.get("type") == "Feature":
        return obj["geometry"]
    return obj  # assume bare geometry

def find_lon_lat(ds):
    pairs = [
        ("lon","lat"),
        ("longitude","latitude"),
        ("XLONG_M","XLAT_M"),
        ("XLONG","XLAT"),
    ]
    for lonn, latn in pairs:
        if (lonn in ds) and (latn in ds):
            lon = ds[lonn]
            lat = ds[latn]
            if "time" in lon.dims: lon = lon.isel(time=0, drop=True)
            if "time" in lat.dims: lat = lat.isel(time=0, drop=True)
            rename = {}
            if "south_north" in lat.dims: rename["south_north"] = "y"
            if "west_east"  in lat.dims: rename["west_east"]  = "x"
            if rename:
                lat = lat.rename(rename)
                lon = lon.rename(rename)
            return lon, lat
    raise KeyError("Could not locate lon/lat in dataset.")

def guess_precip_var(ds):
    cand = set(ds.data_vars)
    for k in ["PREC_ACC_NC", "APCP", "TP", "TOT_PREC", "pr", "precip"]:
        if k in cand:
            return k
    if {"RAINNC","RAINC"} <= cand: return "RAINNC+RAINC"
    for k in ["RAINNC","RAINC","PREC_ACC_C"]:
        if k in cand:
            return k
    raise KeyError("No recognizable precip variable found.")

def to_increments_mm(da, time_dim="time"):
    """
    Make per-step increments and ensure units are mm.
    Handles cumulative-with-resets and common unit conventions (kg m^-2, meters).
    """
    units = str(da.attrs.get("units", "")).lower()

    # kg m^-2 == mm
    if ("kg" in units and "m-2" in units) or ("kg m" in units and "-2" in units):
        da = da.copy()
        da.attrs["units"] = "mm"

    # meters -> mm (common for WRF accum)
    if units.strip() in {"m", "meter", "meters"}:
        da = da * 1000.0
        da = da.copy()
        da.attrs["units"] = "mm"

    # Decide cumulative vs already-incremental from a tiny slice
    probe = da
    for d in da.dims:
        if d != time_dim:
            probe = probe.isel({d: 0})
    if probe.sizes.get(time_dim, 0) > 3:
        s = np.asarray(probe.values).squeeze()
        dif = np.diff(s)
        is_accum = (np.count_nonzero(dif < -1e-6) / max(1, dif.size)) < 0.05
    else:
        is_accum = True  # conservative

    if is_accum or "acc" in (da.name or "").lower():
        da = da.sortby(time_dim)
        inc = da.diff(time_dim, label="upper")
        inc = inc.where(inc >= 0, 0).fillna(0)
        inc.attrs["units"] = "mm"
        return inc

    # already per-step
    da = da.copy()
    if da.attrs.get("units", "").lower() not in {"mm", "millimeter", "millimeters"}:
        # last resort: assume mm-like if in kg m^-2 or already handled above
        da.attrs["units"] = "mm"
    return da

def build_mask(lon, lat, geom):
    import geopandas as gpd, regionmask
    from shapely.geometry import shape
    poly = shape(geom)
    gdf = gpd.GeoDataFrame({"name": ["Skagit"]}, geometry=[poly], crs="EPSG:4326")
    try:
        regs = regionmask.Regions.from_geopandas(gdf, names="name", name="basin")
        m = regs.mask(lon, lat)  # NaN outside, 0 inside
        mask = (~m.isnull())
    except AttributeError:
        m = regionmask.mask_geopandas(gdf, lon, lat)
        mask = (~m.isnull()) if isinstance(m, xr.DataArray) else xr.DataArray(
            np.where(np.isnan(m), False, True), dims=lat.dims, coords=lat.coords
        )
    if mask.sum() == 0:
        raise RuntimeError("Polygon produced empty mask on this grid.")
    return mask

# ---------- Defensive Intake walk (skip parquet / missing plugins) ----------
def _safe_list(cat):
    try:
        return list(cat)
    except Exception:
        return []

def iter_entries_safe(cat, prefix=""):
    """
    Yield (path_string, entry) for nested sub-catalogs without forcing plugin loads.
    Skips entries whose driver clearly needs a missing plugin (e.g., parquet).
    """
    for key in _safe_list(cat):
        try:
            entry = cat[key]
        except Exception:
            continue

        path = f"{prefix}{key}"
        drv = (getattr(entry, "_driver", "") or "").lower()
        if "parquet" in drv:
            continue

        try:
            if entry.container == "catalog":
                subcat = entry
                for subk in _safe_list(subcat):
                    try:
                        subentry = subcat[subk]
                    except Exception:
                        continue
                    subpath = f"{path}:{subk}"
                    subdrv = (getattr(subentry, "_driver", "") or "").lower()
                    if "parquet" in subdrv:
                        continue
                    if getattr(subentry, "container", None) == "catalog":
                        try:
                            subsub = subentry
                            for subsubk in _safe_list(subsub):
                                try:
                                    subsubentry = subsub[subsubk]
                                except Exception:
                                    continue
                                subsubpath = f"{subpath}:{subsubk}"
                                subsubdrv = (getattr(subsubentry, "_driver", "") or "").lower()
                                if "parquet" in subsubdrv:
                                    continue
                                yield subsubpath, subsubentry
                        except Exception:
                            continue
                    else:
                        yield subpath, subentry
            else:
                yield path, entry
        except Exception:
            continue

def open_conus_source_intake_only():
    """Find a CONUS404 precipitation-capable entry via Intake; open with adapter defaults."""
    cat = intake.open_catalog(CAT_URL)
    tried = []
    for path, entry in iter_entries_safe(cat):
        if "conus404" not in path.lower():
            continue

        drv = (getattr(entry, "_driver", "") or "").lower()
        if "parquet" in drv:
            continue

        # IMPORTANT: do NOT pass storage_options here (this adapter rejects it)
        try:
            ds = entry.to_dask()
        except Exception as e:
            tried.append((path, f"open failed: {type(e).__name__}: {e}"))
            continue

        if not hasattr(ds, "data_vars") or "time" not in ds.dims:
            tried.append((path, "opened but not xarray/time"))
            continue

        try:
            pv = guess_precip_var(ds)
            print(f"[source] using intake entry: {path} (precip var: {pv})")
            return ds, pv, f"intake:{path}"
        except Exception:
            tried.append((path, "no precip var"))
            continue

    msg = "No precip-capable CONUS404 source found via intake."
    if tried:
        preview = "\n".join(f" - {p}: {why}" for p, why in tried[:15])
        if len(tried) > 15:
            preview += f"\n ... and {len(tried)-15} more."
        msg += "\nTried:\n" + preview
    raise RuntimeError(msg)

# --- chunk helper: handle both tuple-of-tuples (old) and dict (new) styles
def first_chunk(da: xr.DataArray, dim: str) -> int:
    """
    Return the first chunk size along `dim`, robust to xarray/dask versions.
    """
    # Newer: dict-like .chunksizes or .chunks
    if hasattr(da, "chunksizes") and isinstance(da.chunksizes, dict) and dim in da.chunksizes:
        return int(da.chunksizes[dim][0])
    if hasattr(da, "chunks") and isinstance(da.chunks, dict) and dim in da.chunks:
        return int(da.chunks[dim][0])

    # Older: tuple-of-tuples ordered by axis
    axis = da.get_axis_num(dim)
    ch = getattr(da, "chunks", None)
    if ch is None:
        # fallback: unchunked, use size
        return int(da.sizes[dim])
    return int(ch[axis][0])

# ---------------- Main ----------------
# 1) open
geom = read_geom(BOUNDARY_GEO)
ds, precip_var, src_label = open_conus_source_intake_only()
print(f"opened source: {src_label} | precip var: {precip_var}")

# 2) choose precip
if precip_var == "RAINNC+RAINC":
    p_all = (ds["RAINNC"] + ds["RAINC"]).rename("PREC_TOTAL")
else:
    p_all = ds[precip_var]

# 3) time subset
t0, t1 = f"{YEAR_MIN}-01-01", f"{YEAR_MAX}-12-31"
if "time" not in p_all.dims:
    raise RuntimeError("Dataset has no 'time' dimension.")
p = p_all.sel(time=slice(t0, t1))

# 4) increments → mm
inc = to_increments_mm(p, "time")

# 5) lon/lat + basin mask (weights)
lon, lat = find_lon_lat(ds)
mask = build_mask(lon, lat, geom)
w = mask.astype("float32")
w = w / w.sum()

# 6) daily sums (grid + basin)
daily_grid  = inc.resample(time="1D").sum()
daily_basin = (inc * w).sum(["y","x"], skipna=True).resample(time="1D").sum()

# sanity
ann_mean = float(daily_basin.resample(time="YE").sum().mean().compute())
print(f"[check] Skagit mean annual (CONUS404) ≈ {ann_mean:.1f} mm/yr over {YEAR_MIN}-{YEAR_MAX}")

# 7) uniform chunking for Zarr v2
daily_basin = daily_basin.chunk({"time": 180})
daily_grid  = daily_grid.chunk({"time": 90, "y": 256, "x": 256})
mask_c      = mask.chunk({"y": 256, "x": 256})
lat_c       = lat.chunk({"y": 256, "x": 256})
lon_c       = lon.chunk({"y": 256, "x": 256})

# 8) write basin series FIRST (Zarr v2)
encoding_series = {
    "precip_basin_daily": {
        "chunks": (first_chunk(daily_basin, "time"),),
        "compressor": None,
        "dtype": "float32",
    }
}

if OUT_ZARR.exists():
    shutil.rmtree(OUT_ZARR)

out1 = xr.Dataset(
    {"precip_basin_daily": daily_basin.astype("float32")},
    coords={"time": daily_basin["time"]},
    attrs={
        "source": src_label,
        "precip_var": precip_var,
        "years": f"{YEAR_MIN}-{YEAR_MAX}",
        "note": "precip_basin_daily: Skagit basin-weighted daily total (mm)",
    },
)
out1.to_zarr(OUT_ZARR, mode="w", consolidated=True, zarr_format=2, encoding=encoding_series)
print(f"[write] basin series → {OUT_ZARR}")

# 9) append grid + mask
encoding_grid = {
    "precip_daily": {
        "chunks": (
            first_chunk(daily_grid, "time"),
            first_chunk(daily_grid, "y"),
            first_chunk(daily_grid, "x"),
        ),
        "compressor": None,
        "dtype": "float32",
    },
    "basin_mask": {
        "chunks": (first_chunk(mask_c, "y"), first_chunk(mask_c, "x")),
        "compressor": None,
        "dtype": "float32",
    },
    "lat": {
        "chunks": (first_chunk(lat_c, "y"), first_chunk(lat_c, "x")),
        "compressor": None,
        "dtype": "float32",
    },
    "lon": {
        "chunks": (first_chunk(lon_c, "y"), first_chunk(lon_c, "x")),
        "compressor": None,
        "dtype": "float32",
    },
}

out2 = xr.Dataset(
    data_vars={
        "precip_daily": daily_grid.astype("float32"),
        "basin_mask": mask_c.astype("float32"),
    },
    coords={
        "time": daily_grid["time"],
        "y": lat_c["y"], "x": lat_c["x"],
        "lat": (("y","x"), lat_c.data.astype("float32")),
        "lon": (("y","x"), lon_c.data.astype("float32")),
    },
    attrs={
        "source": src_label,
        "precip_var": precip_var,
        "years": f"{YEAR_MIN}-{YEAR_MAX}",
        "note": "precip_daily: daily sum on native grid (mm); basin_mask: True in Skagit",
    },
)
out2.to_zarr(OUT_ZARR, mode="a", consolidated=True, zarr_format=2, encoding=encoding_grid)
print(f"[write] grid + mask appended → {OUT_ZARR}")

print("[done] All writes completed.")

[source] using intake entry: conus404-catalog:conus404-hourly-osn (precip var: PREC_ACC_NC)
opened source: intake:conus404-catalog:conus404-hourly-osn | precip var: PREC_ACC_NC


In [1]:
import time

In [ ]:
# ========================= CONUS404 → Skagit daily precip (Zarr v2) =========================
# Defensive Intake walk (skip parquet/missing plugins), increments→mm, basin+grid,
# Zarr v2 with uniform chunking, two-phase write (series first, then grid+mask)
# ============================================================================================

import os, json, shutil
from pathlib import Path

import numpy as np
import xarray as xr
import intake

# ---------------- Config  ----------------
CAT_URL      = "https://raw.githubusercontent.com/hytest-org/hytest/main/dataset_catalog/hytest_intake_catalog.yml"
BOUNDARY_GEO = Path("../data/GIS/SkagitBoundary.json")    # GeoJSON Feature / FeatureCollection / raw geometry
OUT_ZARR     = Path("/data0/balaji24/data/derived/conus404_skagit_precip_daily.zarr")

YEAR_MIN, YEAR_MAX = 2014, 2014

# ---------------- Helpers ----------------
def read_geom(path: Path):
    obj = json.load(open(path))
    if isinstance(obj, dict) and obj.get("type") == "FeatureCollection":
        return obj["features"][0]["geometry"]
    if isinstance(obj, dict) and obj.get("type") == "Feature":
        return obj["geometry"]
    return obj  # assume bare geometry

def find_lon_lat(ds):
    pairs = [
        ("lon","lat"),
        ("longitude","latitude"),
        ("XLONG_M","XLAT_M"),
        ("XLONG","XLAT"),
    ]
    for lonn, latn in pairs:
        if (lonn in ds) and (latn in ds):
            lon = ds[lonn]
            lat = ds[latn]
            if "time" in lon.dims: lon = lon.isel(time=0, drop=True)
            if "time" in lat.dims: lat = lat.isel(time=0, drop=True)
            rename = {}
            if "south_north" in lat.dims: rename["south_north"] = "y"
            if "west_east"  in lat.dims: rename["west_east"]  = "x"
            if rename:
                lat = lat.rename(rename)
                lon = lon.rename(rename)
            return lon, lat
    raise KeyError("Could not locate lon/lat in dataset.")

def guess_precip_var(ds):
    cand = set(ds.data_vars)
    for k in ["PREC_ACC_NC", "APCP", "TP", "TOT_PREC", "pr", "precip"]:
        if k in cand:
            return k
    if {"RAINNC","RAINC"} <= cand: return "RAINNC+RAINC"
    for k in ["RAINNC","RAINC","PREC_ACC_C"]:
        if k in cand:
            return k
    raise KeyError("No recognizable precip variable found.")

def to_increments_mm(da, time_dim="time"):
    """
    Make per-step increments and ensure units are mm.
    Handles cumulative-with-resets and common unit conventions (kg m^-2, meters).
    """
    units = str(da.attrs.get("units", "")).lower()

    # kg m^-2 == mm
    if ("kg" in units and "m-2" in units) or ("kg m" in units and "-2" in units):
        da = da.copy()
        da.attrs["units"] = "mm"

    # meters -> mm (common for WRF accum)
    if units.strip() in {"m", "meter", "meters"}:
        da = da * 1000.0
        da = da.copy()
        da.attrs["units"] = "mm"

    # Decide cumulative vs already-incremental from a tiny slice
    probe = da
    for d in da.dims:
        if d != time_dim:
            probe = probe.isel({d: 0})
    if probe.sizes.get(time_dim, 0) > 3:
        s = np.asarray(probe.values).squeeze()
        dif = np.diff(s)
        is_accum = (np.count_nonzero(dif < -1e-6) / max(1, dif.size)) < 0.05
    else:
        is_accum = True  # conservative

    if is_accum or "acc" in (da.name or "").lower():
        da = da.sortby(time_dim)
        inc = da.diff(time_dim, label="upper")
        inc = inc.where(inc >= 0, 0).fillna(0)
        inc.attrs["units"] = "mm"
        return inc

    # already per-step
    da = da.copy()
    if da.attrs.get("units", "").lower() not in {"mm", "millimeter", "millimeters"}:
        # last resort: assume mm-like if in kg m^-2 or already handled above
        da.attrs["units"] = "mm"
    return da

def build_mask(lon, lat, geom):
    import geopandas as gpd, regionmask
    from shapely.geometry import shape
    poly = shape(geom)
    gdf = gpd.GeoDataFrame({"name": ["Skagit"]}, geometry=[poly], crs="EPSG:4326")
    try:
        regs = regionmask.Regions.from_geopandas(gdf, names="name", name="basin")
        m = regs.mask(lon, lat)  # NaN outside, 0 inside
        mask = (~m.isnull())
    except AttributeError:
        m = regionmask.mask_geopandas(gdf, lon, lat)
        mask = (~m.isnull()) if isinstance(m, xr.DataArray) else xr.DataArray(
            np.where(np.isnan(m), False, True), dims=lat.dims, coords=lat.coords
        )
    if mask.sum() == 0:
        raise RuntimeError("Polygon produced empty mask on this grid.")
    return mask

# ---------- Defensive Intake walk (skip parquet / missing plugins) ----------
def _safe_list(cat):
    try:
        return list(cat)
    except Exception:
        return []

def iter_entries_safe(cat, prefix=""):
    """
    Yield (path_string, entry) for nested sub-catalogs without forcing plugin loads.
    Skips entries whose driver clearly needs a missing plugin (e.g., parquet).
    """
    for key in _safe_list(cat):
        try:
            entry = cat[key]
        except Exception:
            continue

        path = f"{prefix}{key}"
        drv = (getattr(entry, "_driver", "") or "").lower()
        if "parquet" in drv:
            continue

        try:
            if entry.container == "catalog":
                subcat = entry
                for subk in _safe_list(subcat):
                    try:
                        subentry = subcat[subk]
                    except Exception:
                        continue
                    subpath = f"{path}:{subk}"
                    subdrv = (getattr(subentry, "_driver", "") or "").lower()
                    if "parquet" in subdrv:
                        continue
                    if getattr(subentry, "container", None) == "catalog":
                        try:
                            subsub = subentry
                            for subsubk in _safe_list(subsub):
                                try:
                                    subsubentry = subsub[subsubk]
                                except Exception:
                                    continue
                                subsubpath = f"{subpath}:{subsubk}"
                                subsubdrv = (getattr(subsubentry, "_driver", "") or "").lower()
                                if "parquet" in subsubdrv:
                                    continue
                                yield subsubpath, subsubentry
                        except Exception:
                            continue
                    else:
                        yield subpath, subentry
            else:
                yield path, entry
        except Exception:
            continue

def open_conus_source_intake_only():
    """Find a CONUS404 precipitation-capable entry via Intake; open with adapter defaults."""
    cat = intake.open_catalog(CAT_URL)
    tried = []
    for path, entry in iter_entries_safe(cat):
        if "conus404" not in path.lower():
            continue

        drv = (getattr(entry, "_driver", "") or "").lower()
        if "parquet" in drv:
            continue

        # IMPORTANT: do NOT pass storage_options here (this adapter rejects it)
        try:
            ds = entry.to_dask()
        except Exception as e:
            tried.append((path, f"open failed: {type(e).__name__}: {e}"))
            continue

        if not hasattr(ds, "data_vars") or "time" not in ds.dims:
            tried.append((path, "opened but not xarray/time"))
            continue

        try:
            pv = guess_precip_var(ds)
            print(f"[source] using intake entry: {path} (precip var: {pv})")
            return ds, pv, f"intake:{path}"
        except Exception:
            tried.append((path, "no precip var"))
            continue

    msg = "No precip-capable CONUS404 source found via intake."
    if tried:
        preview = "\n".join(f" - {p}: {why}" for p, why in tried[:15])
        if len(tried) > 15:
            preview += f"\n ... and {len(tried)-15} more."
        msg += "\nTried:\n" + preview
    raise RuntimeError(msg)

# --- chunk helper: handle both tuple-of-tuples (old) and dict (new) styles
def first_chunk(da: xr.DataArray, dim: str) -> int:
    """
    Return the first chunk size along `dim`, robust to xarray/dask versions.
    """
    # Newer: dict-like .chunksizes or .chunks
    if hasattr(da, "chunksizes") and isinstance(da.chunksizes, dict) and dim in da.chunksizes:
        return int(da.chunksizes[dim][0])
    if hasattr(da, "chunks") and isinstance(da.chunks, dict) and dim in da.chunks:
        return int(da.chunks[dim][0])

    # Older: tuple-of-tuples ordered by axis
    axis = da.get_axis_num(dim)
    ch = getattr(da, "chunks", None)
    if ch is None:
        # fallback: unchunked, use size
        return int(da.sizes[dim])
    return int(ch[axis][0])

# ---------------- Main ----------------
# 1) open
geom = read_geom(BOUNDARY_GEO)
ds, precip_var, src_label = open_conus_source_intake_only()
print(f"opened source: {src_label} | precip var: {precip_var}")

# 2) choose precip
if precip_var == "RAINNC+RAINC":
    p_all = (ds["RAINNC"] + ds["RAINC"]).rename("PREC_TOTAL")
else:
    p_all = ds[precip_var]

# 3) time subset
t0, t1 = f"{YEAR_MIN}-01-01", f"{YEAR_MAX}-12-31"
if "time" not in p_all.dims:
    raise RuntimeError("Dataset has no 'time' dimension.")
p = p_all.sel(time=slice(t0, t1))

# 4) increments → mm
inc = to_increments_mm(p, "time")

# 5) lon/lat + basin mask (weights)
lon, lat = find_lon_lat(ds)
mask = build_mask(lon, lat, geom)
w = mask.astype("float32")
w = w / w.sum()

# 6) daily sums (grid + basin)
daily_grid  = inc.resample(time="1D").sum()
daily_basin = (inc * w).sum(["y","x"], skipna=True).resample(time="1D").sum()

# sanity
ann_mean = float(daily_basin.resample(time="YE").sum().mean().compute())
print(f"[check] Skagit mean annual (CONUS404) ≈ {ann_mean:.1f} mm/yr over {YEAR_MIN}-{YEAR_MAX}")

# 7) uniform chunking for Zarr v2
daily_basin = daily_basin.chunk({"time": 180})
daily_grid  = daily_grid.chunk({"time": 90, "y": 256, "x": 256})
mask_c      = mask.chunk({"y": 256, "x": 256})
lat_c       = lat.chunk({"y": 256, "x": 256})
lon_c       = lon.chunk({"y": 256, "x": 256})

# 8) write basin series FIRST (Zarr v2)
encoding_series = {
    "precip_basin_daily": {
        "chunks": (first_chunk(daily_basin, "time"),),
        "compressor": None,
        "dtype": "float32",
    }
}

if OUT_ZARR.exists():
    shutil.rmtree(OUT_ZARR)

out1 = xr.Dataset(
    {"precip_basin_daily": daily_basin.astype("float32")},
    coords={"time": daily_basin["time"]},
    attrs={
        "source": src_label,
        "precip_var": precip_var,
        "years": f"{YEAR_MIN}-{YEAR_MAX}",
        "note": "precip_basin_daily: Skagit basin-weighted daily total (mm)",
    },
)
out1.to_zarr(OUT_ZARR, mode="w", consolidated=True, zarr_format=2, encoding=encoding_series)
print(f"[write] basin series → {OUT_ZARR}")

# 9) append grid + mask
encoding_grid = {
    "precip_daily": {
        "chunks": (
            first_chunk(daily_grid, "time"),
            first_chunk(daily_grid, "y"),
            first_chunk(daily_grid, "x"),
        ),
        "compressor": None,
        "dtype": "float32",
    },
    "basin_mask": {
        "chunks": (first_chunk(mask_c, "y"), first_chunk(mask_c, "x")),
        "compressor": None,
        "dtype": "float32",
    },
    "lat": {
        "chunks": (first_chunk(lat_c, "y"), first_chunk(lat_c, "x")),
        "compressor": None,
        "dtype": "float32",
    },
    "lon": {
        "chunks": (first_chunk(lon_c, "y"), first_chunk(lon_c, "x")),
        "compressor": None,
        "dtype": "float32",
    },
}

out2 = xr.Dataset(
    data_vars={
        "precip_daily": daily_grid.astype("float32"),
        "basin_mask": mask_c.astype("float32"),
    },
    coords={
        "time": daily_grid["time"],
        "y": lat_c["y"], "x": lat_c["x"],
        "lat": (("y","x"), lat_c.data.astype("float32")),
        "lon": (("y","x"), lon_c.data.astype("float32")),
    },
    attrs={
        "source": src_label,
        "precip_var": precip_var,
        "years": f"{YEAR_MIN}-{YEAR_MAX}",
        "note": "precip_daily: daily sum on native grid (mm); basin_mask: True in Skagit",
    },
)
out2.to_zarr(OUT_ZARR, mode="a", consolidated=True, zarr_format=2, encoding=encoding_grid)
print(f"[write] grid + mask appended → {OUT_ZARR}")

print("[done] All writes completed.")

ModuleNotFoundError: No module named 'intake'